In [1]:
# import necessary libraries
import pandas as pd
import numpy as np
import gdown
import matplotlib.pyplot as plt
from io import StringIO
import nltk
from nltk.tokenize.punkt import PunktSentenceTokenizer, PunktParameters
import random
import re

In [2]:
# specify file path on google drive and the local folder to which it should be saved
file_id = "1DUgfdEhSo425FA2LJbpgFKH8Jr2jAHV_"
url = f"https://drive.google.com/uc?id={file_id}&export=download"
output = "../01_data/parlspeech_dataset.csv"

# download the dataset
# gdown.download(url, output, quiet=False)

# load it as a pandas df and show first rows
# df = pd.read_csv(output)

# read it from the local directory
parlspeech_df = pd.read_csv("../01_data/parlspeech_dataset.csv")

# read in the dictionary df
group_dictionary_df = pd.read_csv("../01_data/groups_dictionary.csv")

In [3]:
# convert date column to datetime and extract month and year
parlspeech_df["date"] = pd.to_datetime(parlspeech_df["date"])
parlspeech_df["month"] = parlspeech_df["date"].dt.month
parlspeech_df["year"] = parlspeech_df["date"].dt.year

# show first rows
parlspeech_df.head()

,date,agenda,speechnumber,speaker,party,party.facts.id,chair,terms,text,parliament,iso3country,month,year
0,1988-11-22,Queen's Speech,1,CHAIR,NaN,NaN,True,74,I have to acquaint the House that this House h...,UK-HouseOfCommons,GBR,11,1988
1,1988-11-22,First Day [Debate On The Address],2,CHAIR,NaN,NaN,True,55,It may be for the convenience of the House if ...,UK-HouseOfCommons,GBR,11,1988
2,1988-11-22,First Day [Debate On The Address],3,Giles Shaw,Con,1567.0,False,2511,"I beg to move,. That an humble Address be pres...",UK-HouseOfCommons,GBR,11,1988
3,1988-11-22,First Day [Debate On The Address],4,John Maples,Con,1567.0,False,1470,I am delighted to second the motion. When I ha...,UK-HouseOfCommons,GBR,11,1988
4,1988-11-22,First Day [Debate On The Address],5,Neil Kinnock,Lab,1516.0,False,2768,I am sure that I speak for the majority of hon...,UK-HouseOfCommons,GBR,11,1988


In [5]:
# reduce the df to only speeches within the relevant electoral cycles and check number of characters
start_date = pd.to_datetime("2010-05-06")
end_date = pd.to_datetime("2019-12-12")
parlspeech_df_subset = parlspeech_df[(parlspeech_df["date"] >= start_date) & (parlspeech_df["date"] <= end_date) &
                                     (parlspeech_df["speaker"] != "CHAIR") & (parlspeech_df["text"].str.len() > 40)].copy()

In [6]:
# reduce the dataset to contain only speeches within parliamentary questions
all_agendas = parlspeech_df["agenda"].unique()
parliamentary_questions_agendas = [agenda for agenda in all_agendas if "questions" in str(agenda).lower()]
parliamentary_questions_df = parlspeech_df_subset[parlspeech_df_subset["agenda"].isin(parliamentary_questions_agendas)].copy()

In [10]:
def clean_text(text):
    # return empty string if the text column is not a string
    if not isinstance(text, str):
        return ""

    # replace misencoded punctuation
    replacements = {
        "Â£": "£",
        "â€œ": "“",
        "â€": "”",
        "â€˜": "‘",
        "â€™": "’",
        "â€“": "–",
        "â€”": "—",
        "â€¦": "…",
        "â€": '"',
        "Ã©": "é",
    }

    for bad, good in replacements.items():
        text = text.replace(bad, good)

    # normalize whitespaces
    text = re.sub(r"\s+", " ", text)

    # remove leading and trailing whitespaces
    return text.strip()


punkt_param = PunktParameters()
abbreviations = ['hon', 'mr', 'mrs', 'dr', 'ms', 'sir', 'prof']  # lowercase
punkt_param.abbrev_types = set(abbreviations)
tokenizer = PunktSentenceTokenizer(punkt_param)

def split_sentences(text, tokenizer=tokenizer):
    if not isinstance(text, str) or not text.strip():
        return []
    return tokenizer.tokenize(text)

# apply cleaning and sentence splitting
parliamentary_questions_df['clean_text'] = parliamentary_questions_df['text'].apply(clean_text)
parliamentary_questions_df['sentences'] = parliamentary_questions_df['clean_text'].apply(split_sentences)

# export the df as a csv
parliamentary_questions_df.to_csv("../01_data/parliamentary_questions_df.csv", index=False)

In [15]:
# get all individual sentences
sentences_df = parliamentary_questions_df[['sentences']].explode('sentences').reset_index(drop=True)
sentences_df = sentences_df.rename(columns={'sentences': 'sentence'})
sentences_df.head()

,sentence
0,What mechanism he plans to use to review the v...
1,We will fundamentally change the way in which ...
2,We will gain maximum value for money for every...
3,May I take this opportunity to welcome my righ...
4,I am sure that Members on both sides of the Ho...


In [17]:
def create_regex_pattern(dictionary_df):
    # get all patterns in the dictionary to look for
    patterns = [word for col in dictionary_df.columns for word in dictionary_df[col] if pd.notna(word)]

    # initialize empty list to store the regex patterns
    regex_patterns = []

    # loop through all patterns
    for pat in patterns:

        # split pattern into words by spaces
        tokens = pat.split()
        regex_parts = []

        # loop through all tokens of the pattern
        for token in tokens:

            # match any word if asterisk is a separate token
            if token == '*':
                regex_parts.append(r'\w+')
            
            # match any prefix to the word
            elif token.startswith('*') and len(token) > 1:
                word = token[1:]
                regex_parts.append(rf'\w*{re.escape(word)}')
            
            # match any suffix to the word
            elif token.endswith('*') and len(token) > 1:
                word = token[:-1]
                regex_parts.append(rf'{re.escape(word)}\w*')

            # if no asterisk, take the word as is
            else:
                regex_parts.append(re.escape(token))

        # join tokens with \s+ to match spaces and append to the list of patterns
        regex_pattern = r'\b' + r'\s+'.join(regex_parts) + r'\b'
        regex_patterns.append(regex_pattern)

    # combine all regex patterns to one with or condition
    combined_regex = "|".join(regex_patterns)

    return combined_regex

# apply function to the groups dictionary
combined_regex = create_regex_pattern(group_dictionary_df)

# apply the compound regex pattern to all sentences
filtered_df = sentences_df[sentences_df["sentence"].str.contains(combined_regex, flags=re.IGNORECASE, regex=True)]

# get all sentences that did not match the regex pattern
unmatched_df = sentences_df[~sentences_df["sentence"].isin(filtered_df["sentence"])]

In [24]:
# randomly sample sentences from both subsets
sample_matches = filtered_df.sample(n=500, random_state=42)
sample_nonmatches = unmatched_df.sample(n=500, random_state=42)

# concatenate the two and mix up randomly
sample_annotations = pd.concat([sample_matches, sample_nonmatches]).sample(frac=1)

In [34]:
# export as a csv file
sample_annotations["sentence"].to_csv("../01_data/sample_annotations.csv", index=False)

In [121]:
sample_annotations["sentence"].iloc[826]

'We are also placing employment brokers across England and Wales to work with small and medium-sized enterprises and regional businesses.'

In [90]:
search_text = sample_annotations["sentence"].iloc[282]
index = parliamentary_questions_df[parliamentary_questions_df["clean_text"].str.contains(search_text)].index[0]
example_speeches = parliamentary_questions_df.loc[index:index+9]
for idx in range(10):
    print(f"Speaker: {example_speeches["speaker"].iloc[idx]} from Party: {example_speeches["party"].iloc[idx]}")
    print(example_speeches["clean_text"].iloc[idx])
    print("-"*100)

Speaker: Dominic Raab from Party: Con
I thank the Secretary of State for that answer. Although a formal teaching qualification may be a bonus, with Ofsted's rigorous new inspection regime and performance-related pay, does he agree that it would be dogmatic in the extreme to force heads to fire 15,000 teachers, regardless of their impact in the classroom, just because they do not hold a piece of paper?
----------------------------------------------------------------------------------------------------
Speaker: Michael Gove from Party: Con
That is a characteristically acute point from my hon. Friend. The most important thing we need to do is ensure that the quality of teaching in our schools is improving. Ofsted tells us that it is, and I am delighted to report that to the House. That is a result of our reforms.
----------------------------------------------------------------------------------------------------
Speaker: Gerry Sutcliffe from Party: Lab
What will the Secretary of State do 